In [23]:
#_Day20_Continuous_Video_Loop_Confidence_Filtered_Player_Tracking`

In [24]:
!pip install -q ultralytics

In [25]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image as PILImage
from ultralytics import YOLO
import pandas as pd
model = YOLO('yolov8n.pt')
NaN = np.nan

In [26]:
cap = cv2.VideoCapture("/content/football_clip.mp4")
ret,frame = cap.read()
ret2, frame2 = cap.read()   # grab the next frame after the one already used

In [42]:
def get_centroids(frame):
  result = model.predict(frame)
  boxes = result[0].boxes.xyxy.cpu().numpy()
  conf = result[0].boxes.conf.cpu().numpy()
  centroid = []

  for box, c in zip(boxes, conf):
    if c < 0.5:
      continue
    x1, y1, x2, y2 = box
    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2
    centroid.append((cx, cy))

  return centroid   # <- add this back

In [46]:
centroid_frame1 = get_centroids(frame)
centroid_frame2 = get_centroids(frame2)
print(centroid_frame1)
print(centroid_frame2)


0: 384x640 7 persons, 145.1ms
Speed: 4.2ms preprocess, 145.1ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 7 persons, 139.1ms
Speed: 3.7ms preprocess, 139.1ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)
[(np.float32(1576.2703), np.float32(1115.5239)), (np.float32(3170.6206), np.float32(1140.373)), (np.float32(2390.5728), np.float32(1116.5941)), (np.float32(2132.1426), np.float32(1121.69)), (np.float32(1107.7478), np.float32(1110.47)), (np.float32(1812.8302), np.float32(1119.2784)), (np.float32(3752.0742), np.float32(1158.3782))]
[(np.float32(1310.1368), np.float32(1180.262)), (np.float32(2138.9731), np.float32(1163.5256)), (np.float32(3153.0117), np.float32(1179.8584)), (np.float32(2374.056), np.float32(1165.1177)), (np.float32(1808.4924), np.float32(1148.9598)), (np.float32(1198.5328), np.float32(1153.611))]


In [47]:
players_position = {}
next_id = 0

for c in centroid_frame1:
  players_position[next_id] = c
  next_id += 1

print(players_position)

{0: (np.float32(1576.2703), np.float32(1115.5239)), 1: (np.float32(3170.6206), np.float32(1140.373)), 2: (np.float32(2390.5728), np.float32(1116.5941)), 3: (np.float32(2132.1426), np.float32(1121.69)), 4: (np.float32(1107.7478), np.float32(1110.47)), 5: (np.float32(1812.8302), np.float32(1119.2784)), 6: (np.float32(3752.0742), np.float32(1158.3782))}


In [48]:
cap = cv2.VideoCapture('/content/football_clip.mp4')   # reopen from frame 0 — the old cap object is exhausted
frame_count = 0
max_frames = 50   # quick test limit — remove once logic is confirmed correct

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_count += 1
    if frame_count > max_frames:
        break

    centroid = get_centroids(frame)
    for c in centroid:
        best_id = None
        best_distance = float("inf")
        for pid, pos in players_position.items():
            d = math.dist(c, pos)
            if d < best_distance:
                best_id = pid
                best_distance = d
        if best_distance < max_distance:
            players_position[best_id] = c
        else:
            players_position[next_id] = c
            next_id += 1

print(players_position)
print(next_id)


0: 384x640 7 persons, 181.6ms
Speed: 5.4ms preprocess, 181.6ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 7 persons, 179.0ms
Speed: 4.5ms preprocess, 179.0ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 7 persons, 160.5ms
Speed: 7.6ms preprocess, 160.5ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 7 persons, 174.2ms
Speed: 5.3ms preprocess, 174.2ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 7 persons, 168.3ms
Speed: 3.7ms preprocess, 168.3ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 7 persons, 165.5ms
Speed: 3.8ms preprocess, 165.5ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 7 persons, 150.7ms
Speed: 5.4ms preprocess, 150.7ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 7 persons, 168.0ms
Speed: 4.9ms preprocess, 168.0ms inference, 1.3ms postprocess per 

In [35]:
# **Day 19-20: Persistent ID Tracking and Continuous Video Loop**

# **Conclusion:**
# - Extended persistent player IDs into a continuous `while` loop, running
#     detection + matching over every frame via `cap.read()` until the video ends
# - Hit a real bug during testing: 80 unique IDs generated over just 50 frames,
#      despite only 6-7 real players — way too many
# - Diagnosed the cause: low-confidence false-positive detections (not camera
#     movement, not crowd) getting treated as new players every frame
# - Fixed it by pulling `boxes.conf` and filtering out detections below a 0.5
#     confidence threshold before computing centroids
# - Reran the test: 80 IDs dropped to 11 — a realistic number for 6-7 real players
#     with occasional occlusion
# - Result: a working continuous player tracker validated on real football footage,
#     ready to scale up to the full video